In [1]:
import os
import re

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import xgboost

import sqlite3

import pickle

In [3]:
conn = sqlite3.connect('Z:\Active_Users_Data\Jillian\DENV_assay_dvlp\CPOutput_DR\Dengue20X_timeseries_CPA_DR.db')

In [6]:
df = pd.read_sql_query('SELECT * FROM MyExpt_Per_Object', conn)

In [7]:
meta_cols = df.columns[df.columns.str.contains(pat='Metadata|ImageNumber|Location|Center|Cells_Number_Object_Number|ObjectNumber', flags=re.IGNORECASE)].tolist()


In [8]:
cols = df.drop(columns=meta_cols).select_dtypes(include='float64').columns.tolist()

In [9]:
data_cols = df[cols].columns[df[cols].columns.str.contains(pat='Epro|NS4B',flags=re.IGNORECASE)].tolist()

In [24]:
with open('data_cols_reduced', 'rb') as f:
    time_cols = pickle.load(f)

In [34]:
print(len(data_cols))
print(len(time_cols))

298
334


In [27]:
common_data_cols = set(data_cols).intersection(time_cols)
common_data_cols = list(common_data_cols)

In [30]:
','.join(common_data_cols)

'Nuclei_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256,Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_00_256,Nuclei_Texture_Entropy_NS4BAfterMath_6_03_256,Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_03_256,Nuclei_Intensity_UpperQuartileIntensity_EproAfterMath,Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_00_256,Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_01_256,Nuclei_Texture_InfoMeas1_EproAfterMath_6_00_256,Cells_Intensity_UpperQuartileIntensity_EproAfterMath,Nuclei_Texture_InfoMeas2_EproAfterMath_6_03_256,Nuclei_Texture_Correlation_EproAfterMath_6_00_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_00_256,Nuclei_Intensity_MaxIntensity

In [43]:
with open('data_cols_reduced_DR', 'wb') as f:
    pickle.dump(common_data_cols, f)

In [29]:
cols = ['ImageNumber', 'Image_Metadata_WellID']+common_data_cols

In [31]:
query = "SELECT ImageNumber, ObjectNumber, Nuclei_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256,Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_00_256,Nuclei_Texture_Entropy_NS4BAfterMath_6_03_256,Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_03_256,Nuclei_Intensity_UpperQuartileIntensity_EproAfterMath,Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_00_256,Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_01_256,Nuclei_Texture_InfoMeas1_EproAfterMath_6_00_256,Cells_Intensity_UpperQuartileIntensity_EproAfterMath,Nuclei_Texture_InfoMeas2_EproAfterMath_6_03_256,Nuclei_Texture_Correlation_EproAfterMath_6_00_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_00_256,Nuclei_Intensity_MaxIntensityEdge_NS4BAfterMath,Cytoplasm_Intensity_MaxIntensityEdge_EproAfterMath,Nuclei_Texture_SumEntropy_EproAfterMath_6_00_256,Nuclei_Intensity_StdIntensity_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_02_256,Cytoplasm_Texture_InfoMeas1_NS4BAfterMath_6_03_256,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_01_256,Nuclei_Intensity_StdIntensityEdge_NS4BAfterMath,Nuclei_Texture_SumVariance_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_00_256,Nuclei_Texture_Variance_EproAfterMath_6_01_256,Cytoplasm_Texture_InfoMeas1_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_Contrast_NS4BAfterMath_6_01_256,Nuclei_Texture_DifferenceEntropy_NS4BAfterMath_6_00_256,Cytoplasm_Texture_Correlation_EproAfterMath_6_02_256,Cytoplasm_Texture_AngularSecondMoment_NS4BAfterMath_6_03_256,Cytoplasm_Intensity_StdIntensityEdge_EproAfterMath,Nuclei_Texture_Correlation_EproAfterMath_6_01_256,Nuclei_Texture_InverseDifferenceMoment_NS4BAfterMath_6_01_256,Cytoplasm_Intensity_MeanIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_SumVariance_EproAfterMath_6_00_256,Nuclei_Intensity_MeanIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_02_256,Cytoplasm_Texture_AngularSecondMoment_NS4BAfterMath_6_02_256,Cells_Intensity_UpperQuartileIntensity_NS4BAfterMath,Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_00_256,Cytoplasm_Texture_Correlation_NS4BAfterMath_6_00_256,Cytoplasm_Texture_SumAverage_EproAfterMath_6_00_256,Nuclei_Texture_SumAverage_NS4BAfterMath_6_03_256,Cytoplasm_Texture_InverseDifferenceMoment_EproAfterMath_6_02_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_02_256,Cytoplasm_Texture_InfoMeas2_EproAfterMath_6_00_256,Cytoplasm_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Texture_DifferenceVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_02_256,Nuclei_Texture_SumAverage_EproAfterMath_6_02_256,Nuclei_Texture_SumVariance_EproAfterMath_6_02_256,Nuclei_Intensity_UpperQuartileIntensity_NS4BAfterMath,Nuclei_Texture_DifferenceVariance_NS4BAfterMath_6_00_256,Nuclei_Texture_SumAverage_NS4BAfterMath_6_00_256,Cytoplasm_Intensity_MADIntensity_EproAfterMath,Nuclei_Texture_AngularSecondMoment_NS4BAfterMath_6_01_256,Cytoplasm_Texture_DifferenceVariance_NS4BAfterMath_6_00_256,Cytoplasm_Texture_Entropy_NS4BAfterMath_6_01_256,Nuclei_Texture_InfoMeas2_EproAfterMath_6_01_256,Nuclei_Intensity_MinIntensity_NS4BAfterMath,Cells_Intensity_MaxIntensity_EproAfterMath,Cytoplasm_Texture_Correlation_EproAfterMath_6_01_256,Cytoplasm_Texture_SumVariance_EproAfterMath_6_01_256,Nuclei_Texture_DifferenceVariance_NS4BAfterMath_6_02_256,Nuclei_Intensity_MinIntensity_EproAfterMath,Nuclei_Texture_InfoMeas1_NS4BAfterMath_6_00_256,Nuclei_Texture_AngularSecondMoment_EproAfterMath_6_01_256,Nuclei_Intensity_LowerQuartileIntensity_NS4BAfterMath,Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_01_256,Nuclei_Texture_Entropy_EproAfterMath_6_02_256,Nuclei_Intensity_MeanIntensityEdge_NS4BAfterMath,Nuclei_Texture_DifferenceVariance_EproAfterMath_6_01_256,Cells_Intensity_MinIntensityEdge_EproAfterMath,Cytoplasm_Texture_Entropy_EproAfterMath_6_01_256,Cytoplasm_Texture_AngularSecondMoment_EproAfterMath_6_00_256,Nuclei_Intensity_MaxIntensity_NS4BAfterMath,Cytoplasm_Texture_AngularSecondMoment_EproAfterMath_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_00_256,Cells_Intensity_StdIntensity_NS4BAfterMath,Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_02_256,Cells_Intensity_IntegratedIntensity_EproAfterMath,Cytoplasm_Texture_AngularSecondMoment_NS4BAfterMath_6_01_256,Cytoplasm_Texture_InfoMeas1_NS4BAfterMath_6_01_256,Nuclei_Texture_DifferenceEntropy_EproAfterMath_6_02_256,Cytoplasm_Texture_Entropy_NS4BAfterMath_6_03_256,Cytoplasm_Intensity_MeanIntensity_EproAfterMath,Cells_Intensity_MeanIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_03_256,Cytoplasm_Intensity_IntegratedIntensityEdge_EproAfterMath,Cytoplasm_Texture_AngularSecondMoment_EproAfterMath_6_01_256,Cytoplasm_Texture_Contrast_NS4BAfterMath_6_03_256,Cells_Intensity_MinIntensity_NS4BAfterMath,Cells_Intensity_MeanIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_SumEntropy_EproAfterMath_6_03_256,Cytoplasm_Texture_Variance_NS4BAfterMath_6_01_256,Nuclei_Intensity_MinIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_InfoMeas2_EproAfterMath_6_01_256,Cytoplasm_Texture_InfoMeas1_NS4BAfterMath_6_02_256,Nuclei_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_InverseDifferenceMoment_EproAfterMath_6_01_256,Cytoplasm_Intensity_UpperQuartileIntensity_EproAfterMath,Nuclei_Texture_Entropy_NS4BAfterMath_6_00_256,Nuclei_Texture_SumEntropy_EproAfterMath_6_02_256,Cytoplasm_Intensity_MaxIntensity_EproAfterMath,Cytoplasm_Texture_Variance_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_02_256,Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_01_256,Nuclei_Texture_Entropy_EproAfterMath_6_03_256,Nuclei_Texture_Variance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_InfoMeas1_EproAfterMath_6_01_256,Cytoplasm_Intensity_MeanIntensityEdge_EproAfterMath,Cytoplasm_Texture_SumEntropy_EproAfterMath_6_01_256,Cytoplasm_Intensity_MinIntensityEdge_NS4BAfterMath,Nuclei_Texture_DifferenceEntropy_EproAfterMath_6_03_256,Nuclei_Texture_AngularSecondMoment_EproAfterMath_6_02_256,Nuclei_Texture_InverseDifferenceMoment_EproAfterMath_6_03_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_01_256,Nuclei_Texture_InfoMeas2_NS4BAfterMath_6_02_256,Cytoplasm_Texture_Correlation_EproAfterMath_6_00_256,Cytoplasm_Texture_DifferenceEntropy_NS4BAfterMath_6_01_256,Nuclei_Texture_SumAverage_NS4BAfterMath_6_01_256,Cytoplasm_Intensity_LowerQuartileIntensity_NS4BAfterMath,Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_01_256,Nuclei_Texture_Entropy_EproAfterMath_6_00_256,Nuclei_Texture_Contrast_EproAfterMath_6_00_256,Cytoplasm_Texture_Variance_NS4BAfterMath_6_02_256,Nuclei_Intensity_MaxIntensityEdge_EproAfterMath,Cytoplasm_Texture_DifferenceVariance_NS4BAfterMath_6_01_256,Nuclei_Texture_AngularSecondMoment_EproAfterMath_6_00_256,Nuclei_Texture_SumAverage_EproAfterMath_6_01_256,Cells_Intensity_StdIntensityEdge_NS4BAfterMath,Nuclei_Texture_SumVariance_NS4BAfterMath_6_00_256,Cells_Intensity_IntegratedIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_SumAverage_NS4BAfterMath_6_02_256,Cells_Intensity_StdIntensity_EproAfterMath,Cytoplasm_Texture_AngularSecondMoment_EproAfterMath_6_02_256,Cytoplasm_Texture_DifferenceEntropy_NS4BAfterMath_6_02_256,Nuclei_Texture_Contrast_EproAfterMath_6_01_256,Nuclei_Intensity_IntegratedIntensityEdge_NS4BAfterMath,Nuclei_Intensity_MedianIntensity_EproAfterMath,Nuclei_Texture_Variance_EproAfterMath_6_02_256,Nuclei_Intensity_MassDisplacement_EproAfterMath,Nuclei_Intensity_MeanIntensityEdge_EproAfterMath,Cells_Intensity_MaxIntensity_NS4BAfterMath,Nuclei_Texture_DifferenceEntropy_NS4BAfterMath_6_03_256,Nuclei_Texture_InfoMeas1_NS4BAfterMath_6_02_256,Cytoplasm_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_Correlation_EproAfterMath_6_03_256,Nuclei_Texture_Correlation_EproAfterMath_6_03_256,Cytoplasm_Texture_SumAverage_EproAfterMath_6_03_256,Nuclei_Texture_InfoMeas1_NS4BAfterMath_6_03_256,Nuclei_Texture_InfoMeas2_NS4BAfterMath_6_03_256,Nuclei_Texture_InverseDifferenceMoment_NS4BAfterMath_6_03_256,Nuclei_Texture_SumAverage_EproAfterMath_6_00_256,Nuclei_Texture_DifferenceEntropy_EproAfterMath_6_01_256,Nuclei_Intensity_MedianIntensity_NS4BAfterMath,Cytoplasm_Texture_InfoMeas2_EproAfterMath_6_02_256,Cytoplasm_Texture_Correlation_NS4BAfterMath_6_03_256,Nuclei_Texture_Entropy_NS4BAfterMath_6_01_256,Nuclei_Texture_Correlation_NS4BAfterMath_6_00_256,Cytoplasm_Intensity_MinIntensity_NS4BAfterMath,Cytoplasm_Texture_SumVariance_NS4BAfterMath_6_00_256,Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_03_256,Nuclei_Texture_InfoMeas1_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Contrast_EproAfterMath_6_02_256,Nuclei_Texture_InfoMeas2_EproAfterMath_6_02_256,Cytoplasm_Texture_InfoMeas2_NS4BAfterMath_6_02_256,Nuclei_Texture_AngularSecondMoment_NS4BAfterMath_6_03_256,Nuclei_Texture_SumVariance_EproAfterMath_6_01_256,Nuclei_Texture_AngularSecondMoment_EproAfterMath_6_03_256,Cytoplasm_Texture_SumVariance_EproAfterMath_6_02_256,Cytoplasm_Texture_SumAverage_NS4BAfterMath_6_01_256,Cells_Intensity_LowerQuartileIntensity_NS4BAfterMath,Nuclei_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Intensity_StdIntensityEdge_EproAfterMath,Nuclei_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MADIntensity_EproAfterMath,Cytoplasm_Intensity_MaxIntensity_NS4BAfterMath,Cells_Intensity_LowerQuartileIntensity_EproAfterMath,Cytoplasm_Intensity_StdIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_Contrast_NS4BAfterMath_6_00_256,Cytoplasm_Texture_Correlation_NS4BAfterMath_6_02_256,Cytoplasm_Texture_InfoMeas1_EproAfterMath_6_00_256,Cells_Intensity_MassDisplacement_NS4BAfterMath,Cytoplasm_Intensity_IntegratedIntensityEdge_NS4BAfterMath,Cytoplasm_Intensity_IntegratedIntensity_EproAfterMath,Cytoplasm_Intensity_IntegratedIntensity_NS4BAfterMath,Cytoplasm_Texture_Contrast_EproAfterMath_6_00_256,Cytoplasm_Texture_InverseDifferenceMoment_EproAfterMath_6_01_256,Cytoplasm_Texture_Variance_EproAfterMath_6_03_256,Nuclei_Texture_SumEntropy_EproAfterMath_6_01_256,Cytoplasm_Texture_AngularSecondMoment_NS4BAfterMath_6_00_256,Cytoplasm_Texture_DifferenceVariance_EproAfterMath_6_00_256,Cytoplasm_Texture_Entropy_NS4BAfterMath_6_00_256,Cytoplasm_Texture_Variance_EproAfterMath_6_00_256,Nuclei_Texture_DifferenceVariance_EproAfterMath_6_02_256,Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_03_256,Cytoplasm_Intensity_LowerQuartileIntensity_EproAfterMath,Cells_Intensity_IntegratedIntensityEdge_EproAfterMath,Cytoplasm_Texture_InfoMeas2_EproAfterMath_6_03_256,Cells_Intensity_MinIntensityEdge_NS4BAfterMath,Nuclei_Texture_SumVariance_EproAfterMath_6_03_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_01_256,Nuclei_Intensity_LowerQuartileIntensity_EproAfterMath,Cytoplasm_Texture_SumEntropy_EproAfterMath_6_02_256,Cytoplasm_Texture_SumAverage_NS4BAfterMath_6_03_256,Nuclei_Texture_DifferenceEntropy_NS4BAfterMath_6_01_256,Cytoplasm_Texture_SumAverage_EproAfterMath_6_02_256,Cells_Intensity_MassDisplacement_EproAfterMath,Nuclei_Intensity_MeanIntensity_NS4BAfterMath,Nuclei_Texture_InfoMeas2_EproAfterMath_6_00_256,Cytoplasm_Intensity_StdIntensity_EproAfterMath,Cells_Intensity_MinIntensity_EproAfterMath,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_02_256,Cytoplasm_Intensity_MeanIntensity_NS4BAfterMath,Cytoplasm_Texture_Variance_NS4BAfterMath_6_03_256,Nuclei_Texture_Correlation_EproAfterMath_6_02_256,Cytoplasm_Intensity_MinIntensityEdge_EproAfterMath,Cytoplasm_Texture_Entropy_EproAfterMath_6_00_256,Cytoplasm_Texture_InverseDifferenceMoment_EproAfterMath_6_00_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_02_256,Cytoplasm_Texture_Contrast_EproAfterMath_6_03_256,Nuclei_Texture_InverseDifferenceMoment_NS4BAfterMath_6_02_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_03_256,Cytoplasm_Texture_DifferenceVariance_NS4BAfterMath_6_02_256,Cytoplasm_Texture_SumVariance_NS4BAfterMath_6_01_256,Nuclei_Texture_SumVariance_EproAfterMath_6_00_256,Cytoplasm_Texture_Entropy_NS4BAfterMath_6_02_256,Cytoplasm_Intensity_UpperQuartileIntensity_NS4BAfterMath,Cytoplasm_Texture_SumVariance_EproAfterMath_6_03_256,Cytoplasm_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Texture_AngularSecondMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_Variance_NS4BAfterMath_6_03_256,Cytoplasm_Texture_Variance_EproAfterMath_6_01_256,Nuclei_Intensity_MinIntensityEdge_EproAfterMath,Cytoplasm_Texture_Correlation_NS4BAfterMath_6_01_256,Cytoplasm_Intensity_MedianIntensity_NS4BAfterMath,Nuclei_Texture_AngularSecondMoment_NS4BAfterMath_6_02_256,Nuclei_Texture_SumAverage_EproAfterMath_6_03_256,Nuclei_Texture_Entropy_EproAfterMath_6_01_256,Cytoplasm_Texture_DifferenceEntropy_EproAfterMath_6_02_256,Cytoplasm_Texture_InfoMeas1_EproAfterMath_6_02_256,Nuclei_Texture_InverseDifferenceMoment_EproAfterMath_6_00_256,Nuclei_Texture_DifferenceVariance_EproAfterMath_6_00_256,Nuclei_Texture_InfoMeas1_EproAfterMath_6_02_256,Nuclei_Texture_InverseDifferenceMoment_EproAfterMath_6_02_256,Nuclei_Texture_DifferenceEntropy_NS4BAfterMath_6_02_256,Nuclei_Texture_SumEntropy_EproAfterMath_6_03_256,Cells_Intensity_MeanIntensity_NS4BAfterMath,Cytoplasm_Texture_DifferenceVariance_NS4BAfterMath_6_03_256,Cytoplasm_Intensity_StdIntensity_NS4BAfterMath,Nuclei_Texture_InfoMeas1_EproAfterMath_6_03_256,Cytoplasm_Texture_DifferenceEntropy_NS4BAfterMath_6_00_256,Cytoplasm_Texture_SumEntropy_NS4BAfterMath_6_03_256,Cytoplasm_Texture_SumAverage_EproAfterMath_6_01_256,Nuclei_Intensity_StdIntensity_EproAfterMath,Cytoplasm_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_SumAverage_NS4BAfterMath_6_02_256,Cells_Intensity_StdIntensityEdge_EproAfterMath,Cytoplasm_Intensity_MaxIntensityEdge_NS4BAfterMath,Cytoplasm_Texture_SumVariance_NS4BAfterMath_6_03_256,Cytoplasm_Texture_Contrast_EproAfterMath_6_01_256,Cytoplasm_Intensity_MassDisplacement_EproAfterMath,Nuclei_Texture_Correlation_NS4BAfterMath_6_02_256,Nuclei_Texture_DifferenceEntropy_EproAfterMath_6_00_256,Nuclei_Texture_DifferenceVariance_NS4BAfterMath_6_03_256,Cells_Intensity_MedianIntensity_NS4BAfterMath,Nuclei_Texture_Correlation_NS4BAfterMath_6_01_256,Nuclei_Intensity_IntegratedIntensity_EproAfterMath,Nuclei_Intensity_IntegratedIntensityEdge_EproAfterMath,Nuclei_Texture_Correlation_NS4BAfterMath_6_03_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_03_256,Cytoplasm_Texture_Contrast_NS4BAfterMath_6_02_256,Cytoplasm_Texture_InverseDifferenceMoment_EproAfterMath_6_03_256,Cytoplasm_Texture_SumEntropy_EproAfterMath_6_00_256,Nuclei_Texture_InfoMeas2_NS4BAfterMath_6_01_256,Cytoplasm_Texture_SumAverage_NS4BAfterMath_6_00_256,Nuclei_Texture_DifferenceVariance_EproAfterMath_6_03_256,Nuclei_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Cytoplasm_Intensity_MinIntensity_EproAfterMath,Cytoplasm_Texture_InfoMeas1_NS4BAfterMath_6_00_256,Nuclei_Texture_InfoMeas2_NS4BAfterMath_6_00_256,Cytoplasm_Texture_SumVariance_NS4BAfterMath_6_02_256,Cytoplasm_Texture_DifferenceEntropy_NS4BAfterMath_6_03_256,Nuclei_Intensity_MaxIntensity_EproAfterMath,Cells_Intensity_MeanIntensityEdge_EproAfterMath,Cells_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Texture_Entropy_NS4BAfterMath_6_02_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_03_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_00_256,Nuclei_Intensity_MADIntensity_EproAfterMath,Nuclei_Texture_InfoMeas1_EproAfterMath_6_01_256 From MyExpt_Per_Object"

In [32]:
data = pd.read_sql_query(query, conn)
data.head()

,ImageNumber,ObjectNumber,Nuclei_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256,...,Cytoplasm_Texture_SumVariance_NS4BAfterMath_6_02_256,Cytoplasm_Texture_DifferenceEntropy_NS4BAfterMath_6_03_256,Nuclei_Intensity_MaxIntensity_EproAfterMath,Cells_Intensity_MeanIntensityEdge_EproAfterMath,Cells_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Texture_Entropy_NS4BAfterMath_6_02_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_03_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_00_256,Nuclei_Intensity_MADIntensity_EproAfterMath,Nuclei_Texture_InfoMeas1_EproAfterMath_6_01_256
0,1,1,12.427141,0.119792,0.016312,84.088518,0.000702,0.293004,66.821020,1.008562,...,92.575998,3.449388,0.013565,0.001779,0.004883,7.266297,24.965409,4.732771,0.000931,-0.021018
1,1,2,5.182567,5.115463,0.029740,142.932207,0.001404,0.169827,640.221950,2.865450,...,406.970272,4.793749,0.067582,0.001666,0.000000,9.302456,285.584185,6.201816,0.004486,-0.053201
2,1,3,4.807087,12.594567,0.023819,54.011475,0.008942,0.376594,9.204165,6.034524,...,20.719636,2.872729,0.101579,0.006925,0.007843,5.217539,9.562900,3.516471,0.003746,-0.102264
3,1,4,10.962365,0.000000,0.023362,41.776593,0.000000,0.502320,21.625912,1.482724,...,75.713399,3.128397,0.016007,0.001429,0.000412,6.075225,78.586207,4.173904,0.000000,-0.074483
4,1,5,3.724222,2.838398,0.018463,70.140902,0.001892,0.423455,6.555378,4.647210,...,19.770612,2.315983,0.048829,0.002596,0.002777,3.626230,3.141722,2.542321,0.001816,-0.062699


In [35]:
meta = pd.read_sql_query('SELECT ImageNumber, Image_Metadata_WellID, Image_Metadata_PlateID FROM MyExpt_Per_Image', conn)

In [36]:
df = pd.merge(data, meta, on='ImageNumber')

In [37]:
del data

In [38]:
df.head()

,ImageNumber,ObjectNumber,Nuclei_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256,...,Nuclei_Intensity_MaxIntensity_EproAfterMath,Cells_Intensity_MeanIntensityEdge_EproAfterMath,Cells_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Texture_Entropy_NS4BAfterMath_6_02_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_03_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_00_256,Nuclei_Intensity_MADIntensity_EproAfterMath,Nuclei_Texture_InfoMeas1_EproAfterMath_6_01_256,Image_Metadata_WellID,Image_Metadata_PlateID
0,1,1,12.427141,0.119792,0.016312,84.088518,0.000702,0.293004,66.821020,1.008562,...,0.013565,0.001779,0.004883,7.266297,24.965409,4.732771,0.000931,-0.021018,A01,DR_20221209_143805
1,1,2,5.182567,5.115463,0.029740,142.932207,0.001404,0.169827,640.221950,2.865450,...,0.067582,0.001666,0.000000,9.302456,285.584185,6.201816,0.004486,-0.053201,A01,DR_20221209_143805
2,1,3,4.807087,12.594567,0.023819,54.011475,0.008942,0.376594,9.204165,6.034524,...,0.101579,0.006925,0.007843,5.217539,9.562900,3.516471,0.003746,-0.102264,A01,DR_20221209_143805
3,1,4,10.962365,0.000000,0.023362,41.776593,0.000000,0.502320,21.625912,1.482724,...,0.016007,0.001429,0.000412,6.075225,78.586207,4.173904,0.000000,-0.074483,A01,DR_20221209_143805
4,1,5,3.724222,2.838398,0.018463,70.140902,0.001892,0.423455,6.555378,4.647210,...,0.048829,0.002596,0.002777,3.626230,3.141722,2.542321,0.001816,-0.062699,A01,DR_20221209_143805


In [63]:
model = xgboost.XGBRegressor()
model.load_model('xgb_model_0_48_wDRcontrols')

In [64]:
scaled_data = StandardScaler().fit_transform(df[common_data_cols])

In [65]:
df['score'] = model.predict(scaled_data)

In [66]:
df.head()

,ImageNumber,ObjectNumber,Nuclei_Intensity_MassDisplacement_NS4BAfterMath,Nuclei_Texture_Contrast_EproAfterMath_6_03_256,Cells_Intensity_MaxIntensityEdge_EproAfterMath,Cells_Intensity_IntegratedIntensity_NS4BAfterMath,Cells_Intensity_MedianIntensity_EproAfterMath,Cytoplasm_Texture_InverseDifferenceMoment_NS4BAfterMath_6_00_256,Nuclei_Texture_SumVariance_NS4BAfterMath_6_01_256,Cytoplasm_Texture_Entropy_EproAfterMath_6_03_256,...,Cells_Intensity_MeanIntensityEdge_EproAfterMath,Cells_Intensity_MADIntensity_NS4BAfterMath,Nuclei_Texture_Entropy_NS4BAfterMath_6_02_256,Nuclei_Texture_Contrast_NS4BAfterMath_6_03_256,Nuclei_Texture_SumEntropy_NS4BAfterMath_6_00_256,Nuclei_Intensity_MADIntensity_EproAfterMath,Nuclei_Texture_InfoMeas1_EproAfterMath_6_01_256,Image_Metadata_WellID,Image_Metadata_PlateID,score
0,1,1,12.427141,0.119792,0.016312,84.088518,0.000702,0.293004,66.821020,1.008562,...,0.001779,0.004883,7.266297,24.965409,4.732771,0.000931,-0.021018,A01,DR_20221209_143805,0.857886
1,1,2,5.182567,5.115463,0.029740,142.932207,0.001404,0.169827,640.221950,2.865450,...,0.001666,0.000000,9.302456,285.584185,6.201816,0.004486,-0.053201,A01,DR_20221209_143805,1.274148
2,1,3,4.807087,12.594567,0.023819,54.011475,0.008942,0.376594,9.204165,6.034524,...,0.006925,0.007843,5.217539,9.562900,3.516471,0.003746,-0.102264,A01,DR_20221209_143805,1.000697
3,1,4,10.962365,0.000000,0.023362,41.776593,0.000000,0.502320,21.625912,1.482724,...,0.001429,0.000412,6.075225,78.586207,4.173904,0.000000,-0.074483,A01,DR_20221209_143805,1.146249
4,1,5,3.724222,2.838398,0.018463,70.140902,0.001892,0.423455,6.555378,4.647210,...,0.002596,0.002777,3.626230,3.141722,2.542321,0.001816,-0.062699,A01,DR_20221209_143805,0.939378


In [71]:
df.shape

(815706, 303)

In [72]:
df.to_sql('timemodel_scored_objectlevel', conn, index=False)

815706

In [67]:
data = {
    'ImageNumber':[],
    'Image_Metadata_WellID':[],
    'Image_Metadata_PlateID':[],
    'score':[]
}


for w in df['Image_Metadata_WellID'].unique():
    temp = df.loc[df['Image_Metadata_WellID']==w]
    data['ImageNumber'].append(temp.iloc[0]['ImageNumber'])
    data['Image_Metadata_WellID'].append(temp.iloc[0]['Image_Metadata_WellID'])
    data['Image_Metadata_PlateID'].append(temp.iloc[0]['Image_Metadata_PlateID'])
    data['score'].append(temp['score'].mean())

In [68]:
well_level = pd.DataFrame(data)

In [69]:
well_level.shape

(324, 4)

In [70]:
well_level.to_csv("DR_time_score_0_48_DRcontrols.csv")

In [73]:
conn.close()